# Terminal Case Study — Phase 1: Single-Gate Model

**Case study**: Intermodal Container Terminal | **Phase**: 1 of 5

## Learning Objectives
By the end of this notebook you will be able to:
1. Build and run a single-gate SimPy model using `simdes`.
2. Verify simulation output against the M/M/1 analytical formula.
3. Interpret truck wait time and gate utilisation.
4. Explain the difference between trucks arriving per hour versus per minute.

---
> Phase 1 models only the truck arrival and gate inspection process.
> The crane/yard stage is added in later phases.

In [ ]:
import sys
from pathlib import Path
# Ensure course/ is on sys.path so the case_studies package is importable
_root = next(p for p in [Path.cwd()] + list(Path.cwd().parents) if (p / 'simdes').is_dir())
_course = _root / 'course'
if str(_course) not in sys.path:
    sys.path.insert(0, str(_course))


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from case_studies.terminal.terminal_model import run_terminal, TerminalParams
from simdes.analysis import confidence_interval

## System Description

Trucks arrive at the terminal gate at rate λ = 10 trucks/hour.
A single gate agent inspects each truck; inspection takes an average of 5 minutes.

| Parameter | Value | Unit |
|---|---|---|
| λ | 10/hr = 1/6 per min | trucks/min |
| μ | 1/5 per min | trucks/min |
| ρ = λ/μ | (1/6)/(1/5) = 0.833 | — |
| Theoretical Wq (M/M/1) | ρ/(μ−λ) = 0.833/(0.200−0.167) = 25 min | minutes |

**Simplification for Phase 1**: only the gate stage is modelled (`n_cranes=100`, `crane_mean≈0`).

In [ ]:
# Phase 1: single gate, unlimited cranes (not the focus)
p1 = TerminalParams(
    n_gates=1,
    n_cranes=100,      # effectively unlimited
    arrival_rate=10.0,  # trucks per hour
    gate_mean=5.0,     # minutes
    crane_mean=0.1,    # near-zero
    sim_time=8.0,      # hours
)

df = run_terminal(p1, n_reps=30)
df.head()

In [ ]:
# Analytical M/M/1 benchmark
lam = 10.0 / 60.0   # per minute
mu  = 1.0  / 5.0    # per minute
rho = lam / mu
Wq_theory = rho / (mu - lam)

sim_mean, ci_lo, ci_hi = confidence_interval(df['mean_wait_gate'].to_numpy())

print(f'ρ = {rho:.3f}')
print(f'Theoretical Wq = {Wq_theory:.2f} min')
print(f'Simulation Wq:  {sim_mean:.2f} min  95% CI [{ci_lo:.2f}, {ci_hi:.2f}]')
print(f'Theory in CI?   {ci_lo <= Wq_theory <= ci_hi}')

In [ ]:
# Sweep arrival rate: impact on gate wait time
arrival_rates = [6.0, 8.0, 10.0, 11.0, 11.5]   # trucks/hour
rows = []

for arr in arrival_rates:
    p = TerminalParams(n_gates=1, n_cranes=100, arrival_rate=arr,
                       gate_mean=5.0, crane_mean=0.1, sim_time=80.0)  # longer for stability
    df_r = run_terminal(p, n_reps=20)
    m, lo, hi = confidence_interval(df_r['mean_wait_gate'].to_numpy())
    lam_r = arr / 60.0
    rows.append({'arr': arr, 'rho': lam_r / mu, 'sim_Wq': m, 'ci_lo': lo, 'ci_hi': hi})

sweep = pd.DataFrame(rows)
sweep

In [ ]:
# Hockey-stick plot
rho_grid = np.linspace(0.01, 0.97, 200)
Wq_mm1   = (rho_grid / mu) / (1 - rho_grid)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(rho_grid, Wq_mm1, 'k--', label='M/M/1 theory', lw=1.5)
ax.errorbar(sweep['rho'], sweep['sim_Wq'],
            yerr=[sweep['sim_Wq']-sweep['ci_lo'], sweep['ci_hi']-sweep['sim_Wq']],
            fmt='o', color='tab:orange', capsize=4, label='Simulation (20 reps)')
ax.set_xlabel(r'Gate utilisation $\rho$')
ax.set_ylabel('Mean gate wait $W_q$ (min)')
ax.set_title('Phase 1 — Single gate: simulation vs. M/M/1')
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

## Summary

Single-gate terminal = M/M/1 queue.  Simulation matches theory within CIs.
At ρ = 0.83, trucks wait an average of 25 minutes at the gate alone.
In Phase 2 we add multiple gates to reduce this bottleneck.

## Try It Yourself

1. At what arrival rate does the gate wait time exceed 60 minutes?
2. Change `sim_time=1.0` (one hour). How does this affect CI width?
3. If the gate can process a truck in 4 minutes instead of 5, what is the new maximum
   arrival rate that keeps Wq < 10 minutes?